Tailor dim table containing holiday periods for UHelsinki from 2023 to 2026
This information is gathered both from:
- University website: [UHel teaching period](https://studies.helsinki.fi/instructions/article/academic-year-and-teaching-periods?check_logged_in=1)
- Official Finnish holidays from library `holidays`

In [ ]:
%cd ../../

In [ ]:
import polars as pl
import holidays

In [ ]:
pl.Config.set_tbl_rows(100)

# Craft the dim table

In [ ]:
DATE_START = pl.lit('2023-01-01').str.to_date()
DATE_END = pl.lit('2026-12-31').str.to_date()
years=[2023, 2024, 2025, 2026]

## From University teaching period

In [ ]:
easter = (
    pl
    .from_records([
        {'date_begin': '2023-04-06', 'date_end': '2023-04-12'},

        {'date_begin': '2024-03-28', 'date_end': '2024-04-03'},

        {'date_begin': '2025-04-17', 'date_end': '2025-04-23'},

        {'date_begin': '2026-04-02', 'date_end': '2026-04-08'},
    ])
    .with_columns(
        pl.col('date_begin').str.to_date(),
        pl.col('date_end').str.to_date(),
    )
)  # fmt: skip

summer = (
    pl
    .from_records([
        {'date_begin': '2023-06-01', 'date_end': '2023-09-03'},

        {'date_begin': '2024-06-01', 'date_end': '2024-09-01'},

        {'date_begin': '2025-06-01', 'date_end': '2025-08-31'},

        {'date_begin': '2026-06-01', 'date_end': '2026-08-30'},
    ])
    .with_columns(
        pl.col('date_begin').str.to_date(),
        pl.col('date_end').str.to_date(),
    )
)  # fmt: skip


In [ ]:

df = (
    pl.DataFrame()

    # Create blank dataframe with date
    .with_columns(pl.date_range(DATE_START, DATE_END, '1d').alias('date'))
)

entries_easter = (
    df
    .join(easter, how='cross')
    .filter(
        (pl.col('date') >= pl.col('date_begin'))
        & (pl.col('date') <= pl.col('date_end'))
    )
    .select(
        'date',
        pl.lit(True).alias('is_holiday')
    )
)

entries_summer = (
    df
    .join(summer, how='cross')
    .filter(
        (pl.col('date') >= pl.col('date_begin'))
        & (pl.col('date') <= pl.col('date_end'))
    )
    .select(
        'date',
        pl.lit(True).alias('is_holiday')
    )
)

entries = pl.concat([entries_easter, entries_summer])


dim_holidays_uhelsinki = (
    df
    .join(entries, on='date', how='left')
)

dim_holidays_uhelsinki.head()

## From lib `holidays`

In [ ]:
fin_holidays = holidays.Finland(years=years)
dim_holiday_raw = pl.from_records([
    {'date': d, 'name_holiday': n}
    for d, n in fin_holidays.items()
])


dim_holidays_uhelsinki = (
    dim_holidays_uhelsinki

    # Add dim holiday
    .join(dim_holiday_raw, on='date', how='left')
    .with_columns(
        # pl.col('name_holiday').is_not_null().cast(pl.Int32).alias('is_holiday')
        pl.coalesce(
            pl.col('is_holiday'),
            pl.col('name_holiday').is_not_null()
        ).alias('is_holiday_new')
    )
    .select('date', pl.col('is_holiday_new').alias('is_holiday'))
)

dim_holidays_uhelsinki.head()

## Check

In [ ]:
days_easter_2023 = (
    dim_holidays_uhelsinki
    .filter(
        (1 == 1)
        & (pl.col('date').dt.year() == 2023)
        & (pl.col('date').dt.month() == 4)
        & (pl.col('is_holiday'))
    )
    # .select(pl.col('date').dt.day())
    ['date'].dt.day().to_list()
)

assert days_easter_2023 == [6, 7, 8, 9, 10, 11, 12]

assert dim_holidays_uhelsinki['date'].min().strftime(r"%Y%m%d") == "20230101"
assert dim_holidays_uhelsinki['date'].max().strftime(r"%Y%m%d") == "20261231"

# Save

In [ ]:
path = "data/processed/dim_holidays_uhelsinki.xlsx"
dim_holidays_uhelsinki.write_excel(path)